In [2]:
!pip install -qU transformers datasets peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 24.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType
import torch
from datasets import load_dataset

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
DATASET_NAME = "kritsadaK/EDGAR-CORPUS-Financial-Summarization"
OUTPUT_DIR = "./llama-qlora-summarizer"

In [5]:
ds = load_dataset(DATASET_NAME)

README.md: 0.00B [00:00, ?B/s]

EDGAR-CORPUS-Financial-Summarization.csv:   0%|          | 0.00/794M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10610 [00:00<?, ? examples/s]

Map:   0%|          | 0/10610 [00:00<?, ? examples/s]

KeyError: 'text'

In [15]:
new_ds = ds["train"].remove_columns("model")

In [17]:
SYSTEM_PROMPT = "You are a concise and factual financial document summarizer."
USER_PROMPT = "Summarize the following text:\n{chunk}"

def format_llama3(example):
    sys_part = f"<<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
    user_part = USER_PROMPT.format(chunk=example["input"])

    # full prompt for decoder-only causal LM
    prompt = f"<s>[INST] {sys_part}{user_part} [/INST]"

    return {
        "text": prompt,
        "labels": example["summary"]
    }

In [19]:
mapped_ds = new_ds.map(format_llama3, remove_columns=new_ds.column_names)

Map:   0%|          | 0/10610 [00:00<?, ? examples/s]

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 2048

def tokenize_function(examples):
    # Tokenize input prompt
    model_inputs = tokenizer(
        examples["text"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length"
    )

    # Tokenize labels (targets)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["labels"],
            max_length=MAX_LEN,
            truncation=True,
            padding="max_length"
        )["input_ids"]

    model_inputs["labels"] = labels
    return model_inputs

In [21]:
tokenized_datasets = mapped_ds.map(tokenize_function, batched=True, remove_columns=["text", "labels"])

Map:   0%|          | 0/10610 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [23]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],  # common LLaMA modules
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


In [27]:
# Data collator pads dynamically for variable lengths
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Training args
training_args = TrainingArguments(
    output_dir="./llama3-fin-sum-qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    logging_steps=50,
    num_train_epochs=2,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    save_total_limit=2,
    gradient_checkpointing=True,
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    tokenizer=tokenizer,
    data_collator=data_collator
)

/tmp/ipython-input-1110514080.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [28]:
trainer.train()

# Save adapters for HF
model.save_pretrained("./llama3-fin-sum-lora")

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [24]:
tokenized_datasets

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 10610
})